In [ ]:
import numpy as np
import pandas as pd
import flowio
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

In [11]:
# fcs = flowio.FlowData("W96_E9_E09_057.fcs")
fcs = flowio.FlowData("W96_D11_D11_055.fcs")

n_channels = len(fcs.channels)
events = np.asarray(fcs.events).reshape(-1, n_channels)
df = pd.DataFrame(events, columns=fcs.channels)

# df.to_excel("dupa.xlsx")

print(fcs.pnn_labels)
print(df.head())

['FSC-A', 'FSC-H', 'SSC-A', 'PE-Cy7-A', 'FITC-A', 'APC-A', 'V450-A', 'Time']
              1        2             3          4            5           6  \
0  17230.400391  13365.0  69120.000000  76.799995   669.440002   29.439999   
1   6482.399902   5400.0  41504.000000  87.040001   545.279968  -16.639999   
2  13553.600586  10631.0  71339.515625  64.000000  1191.679932  101.119995   
3  67210.398438  49590.0  63548.160156  40.959999   325.119995  102.399994   
4  66995.203125  49573.0  44591.359375  19.199999   128.000000  -11.520000   

            7          8  
0   66.959999  51.500000  
1  173.910004  51.500000  
2  277.140015  51.500000  
3  156.240005  51.500000  
4  187.860001  51.599998  


In [ ]:
# channel_names = []
# for i in range(fcs.channel_count):
#     # Try marker name first
#     marker_name = fcs.channels[i].name   # usually $PnS
#     if marker_name is None or marker_name == "":
#         # fallback to parameter name $PnN
#         marker_name = fcs.channels[i].short_name
#     channel_names.append(marker_name)

# print(channel_names)
for channel in fcs.channels.values():
    print(channel['pnn'])
print(fcs.channels)

In [ ]:
%matplotlib widget

plt.close('all')

fscatters = df.iloc[:, :2]

x = fscatters.iloc[:, 0]
y = fscatters.iloc[:, 1]

xmax, ymax = 250000, 250000

bins = 1000

heatmap, xedges, yedges = np.histogram2d(x, y, bins=bins, range=[[0, xmax], [0, ymax]])
heatmap = heatmap / heatmap.max()

plt.figure()
hm = sns.heatmap(heatmap.T, cmap='viridis')
hm.invert_yaxis()

plt.show()

heatmap_fig = plt.figure()
hm = sns.heatmap(heatmap.T, cmap='viridis')
hm.invert_yaxis()

plt.show()

In [ ]:
# print(df.mean())
# print(df.median())

# print(x)

x_vals = x.to_numpy().reshape(-1,1)
print(x_vals, type(x_vals), x_vals.shape)

In [ ]:
from sklearn.preprocessing import SplineTransformer
from sklearn.pipeline import make_pipeline

# Use a spline with 4-5 "knots" (points where the curve can bend)
model = make_pipeline(SplineTransformer(n_knots=5, degree=3), LinearRegression())
model.fit(x_vals, y)

# x_fit = x
# y_fit = model.predict(x_fit)

x_smooth = np.linspace(x.min(), x.max(), 1000).reshape(-1, 1)
y_smooth = model.predict(x_smooth)

print(type(x_smooth))

plt.figure()
# plt.plot(x_fit, y_fit)
plt.plot(x_smooth, y_smooth)
plt.scatter(x,y, color='r')

plt.xlim(0, 250000)
plt.ylim(0, 250000)

In [ ]:
plt.close()

plt.figure()
plt.imshow(heatmap.T, origin='lower', aspect='auto', extent=(0,250000,0,250000))
plt.plot(x_smooth, y_smooth, linewidth=2)
plt.xlim(0,250000)
plt.ylim(0,250000)
plt.show()

plt.figure()
hm = sns.heatmap(heatmap.T, cmap='viridis')
hm.invert_yaxis()
plt.plot(x_smooth, y_smooth, linewidth=3)

plt.show()

In [ ]:
from scipy.spatial.distance import cdist
from scipy.interpolate import interp1d

points = fscatters.iloc[:, :].to_numpy()
curve = np.column_stack((x_smooth, y_smooth))

distances = cdist(points, curve)

min_dist = distances.min(axis=1)


# print(min_dist)

curve_interp = interp1d(curve[:, 0],curve[:, 1], kind='linear', fill_value='extrapolate')

y_curve_interpolated_points = curve_interp(fscatters.iloc[:, 0])

# print(y_curve_interpolated_points, len(y_curve_interpolated_points))

mask = (fscatters.iloc[:, 1] > y_curve_interpolated_points) | (min_dist <= 1000)

# filtered = fscatters[min_dist <= 1000]
filtered = fscatters[mask]

plt.close()

plt.figure()
plt.imshow(heatmap.T, origin='lower', aspect='auto', extent=(0,250000,0,250000))
plt.plot(x_smooth, y_smooth, 'r')
plt.scatter(filtered.iloc[:, 0], filtered.iloc[:, 1])
plt.xlim(0,250000)
plt.ylim(0,250000)
plt.show()

In [ ]:
# Draw new heatmap

heatmap_new, xedges, yedges = np.histogram2d(filtered.iloc[:, 0], filtered.iloc[:, 1], bins=bins, range=[[0, xmax], [0, ymax]])
heatmap_new = heatmap_new / heatmap_new.max()

plt.figure()
hm_new = sns.heatmap(heatmap_new.T, cmap='viridis')
hm_new.invert_yaxis()